# Initial Exploration: Article Master Data

Starter notebook for exploring `ie_maestra_articulos.csv` while keeping the large source file out of Git.

## Goals

- Load the CSV with the correct delimiter.
- Preserve codes with leading zeroes.
- Review dimensions, missing values, duplicates, and sector distribution.
- Provide starter cells for searching and exploring article descriptions.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

In [ ]:
DATA_PATH = Path("ie_maestra_articulos.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Place the CSV in the repository root before running this notebook."
    )

df = pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8",
    dtype={"idarticu": "string", "idsector": "string"},
)

df.shape

In [ ]:
df.head(10)

In [ ]:
df.info(memory_usage="deep")

## Basic Data Quality

In [ ]:
summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "n_missing": df.isna().sum(),
        "pct_missing": df.isna().mean().mul(100).round(2),
        "n_unique": df.nunique(dropna=True),
    }
)

summary

In [ ]:
duplicate_id_count = df.duplicated("idarticu").sum()
duplicate_row_count = df.duplicated().sum()

duplicate_id_count, duplicate_row_count

In [ ]:
df.loc[df.duplicated("idarticu", keep=False)].sort_values("idarticu").head(20)

## Sectors

In [ ]:
sector_counts = (
    df.groupby(["idsector", "desc_sector"], dropna=False)
    .size()
    .reset_index(name="n_articles")
    .sort_values("n_articles", ascending=False)
)

sector_counts

In [ ]:
ax = sector_counts.head(20).sort_values("n_articles").plot.barh(
    x="desc_sector",
    y="n_articles",
    figsize=(10, 7),
    legend=False,
)
ax.set_title("Top 20 sectors by number of articles")
ax.set_xlabel("Articles")
ax.set_ylabel("Sector")
plt.tight_layout()

## Article Descriptions

In [ ]:
df["article_description_length"] = df["desc_larga_articulo"].astype("string").str.len()
df["article_description_length"].describe()

In [ ]:
df.nlargest(20, "article_description_length")[["idarticu", "desc_larga_articulo", "idsector", "desc_sector", "article_description_length"]]

In [ ]:
def search_articles(text, n=20):
    mask = df["desc_larga_articulo"].astype("string").str.contains(text, case=False, na=False, regex=False)
    return df.loc[mask, ["idarticu", "desc_larga_articulo", "idsector", "desc_sector"]].head(n)

search_articles("CARREFOUR")

## Possible Next Steps

- Normalize article descriptions for brand or product searches.
- Check whether `idarticu` uniquely identifies each row.
- Join this master data with tickets, sales, or customer data when those datasets are available.